# Step 2: Train Word-Based Sparse MoE (Switch Transformer)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import random
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import wandb
from config import *

# Word vocabulary
all_words = set()
with open("data/train.jsonl", "r") as f:
    for line in f:
        data = json.loads(line)
        all_words.update(data["sentence"])
        all_words.add(data["next_word"])

word_to_idx = {w: i for i, w in enumerate(sorted(all_words)[:VOCAB_SIZE])}
idx_to_word = {i: w for w, i in word_to_idx.items()}

class WordDataset(Dataset):
    def __init__(self, path):
        self.samples = []
        with open(path, "r") as f:
            for line in f:
                self.samples.append(json.loads(line))
    
    def __len__(self): return len(self.samples)
    
    def __getitem__(self, idx):
        s = self.samples[idx]
        input_ids = [word_to_idx.get(w, 0) for w in s["sentence"]]
        target_id = word_to_idx.get(s["next_word"], 0)
        return torch.tensor(input_ids), torch.tensor(target_id), s["domain"]

# Expert (0.5B each)
class Expert(nn.Module):
    def __init__(self, expert_id):
        super().__init__()
        self.expert_id = expert_id
        # 0.5B params: 6 layers x 768 x 3072
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(EMBED_DIM, 12, 3072, batch_first=True)
            for _ in range(6)
        ])
        self.output_proj = nn.Linear(EMBED_DIM, VOCAB_SIZE)
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.output_proj(x)

# Sparse MoE (Switch Transformer)
class SparseMoE(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, EMBED_DIM)
        self.router = nn.Sequential(
            nn.Linear(EMBED_DIM, 256),
            nn.ReLU(),
            nn.Linear(256, NUM_EXPERTS)
        )
        self.experts = nn.ModuleList([Expert(i) for i in range(NUM_EXPERTS)])
        self.usage_counts = torch.zeros(NUM_EXPERTS)
    
    def forward(self, x, domain=None):
        x = self.embed(x)  # [batch, seq_len, embed_dim]
        
        # Router with noise (prevent collapse)
        router_logits = self.router(x.mean(dim=1))  # [batch, NUM_EXPERTS]
        if self.training:
            router_logits += torch.randn_like(router_logits) * 0.01
        
        router_probs = F.softmax(router_logits, dim=-1)
        top1_idx = router_probs.argmax(dim=-1)  # [batch]
        top1_weight = router_probs.max(dim=-1).values
        
        # Update usage counts for load balancing
        for i in range(NUM_EXPERTS):
            self.usage_counts[i] = (top1_idx == i).sum().item()
        
        # Sparse forward (only active expert)
        outputs = torch.zeros(x.size(0), VOCAB_SIZE)
        for i in range(NUM_EXPERTS):
            mask = (top1_idx == i)
            if mask.any():
                expert_out = self.experts[i](x[mask])
                # Take last token prediction
                outputs[mask] = expert_out[:, -1, :]
        
        return outputs, top1_idx, router_probs
    
    def get_load_balancing_loss(self):
        target = torch.ones(NUM_EXPERTS) / NUM_EXPERTS
        actual = self.usage_counts / self.usage_counts.sum()
        return F.kl_div(actual.log(), target, reduction="batchmean")

# Curriculum scheduler
class CurriculumScheduler:
    def __init__(self):
        self.stage = 0
    
    def get_active_domains(self, step):
        stage = min(step // STAGE_STEPS, NUM_EXPERTS - 1)
        if stage == 0:
            return [0]  # only chat
        elif stage == 1:
            return [0, 4]  # chat + software
        else:
            return list(range(stage + 1))

# Initialize
model = SparseMoE()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, TOTAL_STEPS)
dataset = WordDataset("data/train.jsonl")
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
curriculum = CurriculumScheduler()

# Training loop
wandb.init(project="word-moe-llm")
model.train()
global_step = 0

for epoch in range(5):
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}")
    for input_ids, targets, domains in pbar:
        active_domains = curriculum.get_active_domains(global_step)
        
        outputs, selected_experts, router_probs = model(input_ids)
        
        # Loss
        ce_loss = F.cross_entropy(outputs, targets)
        balance_loss = model.get_load_balancing_loss()
        loss = ce_loss + 0.01 * balance_loss
        
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
        # Logging
        wandb.log({
            "loss": loss.item(),
            "ce_loss": ce_loss.item(),
            "balance_loss": balance_loss.item(),
            "lr": scheduler.get_last_lr()[0],
            "step": global_step
        })
        
        pbar.set_postfix(loss=loss.item(), balance=balance_loss.item())
        global_step += 1
        
        if global_step >= TOTAL_STEPS:
            break
    if global_step >= TOTAL_STEPS:
        break

torch.save(model.state_dict(), "models/moe_5b_final.pt")
print("✅ Training complete! Saved to models/moe_5b_final.pt")